# NSL23 Exploratory Isolated-Clip Recognition (CNN + Transformer) — Legacy Colab Notebook

This notebook trains and evaluates a **hybrid CNN–Transformer** model for **isolated Nepali Sign Language (NSL) alphabet recognition** using the **NSL23 dataset** from Zenodo (record **10478554**).  
Dataset record: https://zenodo.org/records/10478554 (CC-BY 4.0)

✅ What you get:
- Automatic **dataset download** + **unzip**
- Robust **indexing** of `.mov/.MOV` videos and **label extraction** from filenames
- Train/val/test split (optionally **grouped by signer**)
- CNN backbone (MobileNetV2) + Transformer encoder (PyTorch)
- Training loop with AMP, checkpointing
- Evaluation: accuracy + per-class F1 + confusion matrix
- Inference helper for a single video

> Note: NSL23 is large (multiple ZIPs). If you want a quick run, you can download only **Part 1** first, then expand.


In [ ]:
# =========================
# 1) Environment & Imports
# =========================
!nvidia-smi -L || true

import os, re, math, json, random, shutil
from pathlib import Path
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import cv2
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score


GPU 0: Tesla T4 (UUID: GPU-86de1784-39d6-4b07-9f44-0c978026c47a)


In [ ]:
# Reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 2) Download NSL23 from Zenodo

Zenodo record lists these files:
- `NSL_Consonant_Part_1.zip` (3.5 GB)
- `NSL_Consonant_Part_2.zip` (3.8 GB)
- `NSL_Consonant_Part_3.zip` (2.5 GB)
- `NSL_Vowel.zip` (0.62 GB)

We download via direct links:
`https://zenodo.org/records/10478554/files/<FILENAME>?download=1`

You can choose which parts to download.


In [ ]:
# =========================
# 2) Download & Unzip
# =========================
DATA_ROOT = Path("/content/nsl23")  # change if you want
DATA_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(DATA_ROOT)

RECORD_ID = "10478554"
BASE = f"https://zenodo.org/records/{RECORD_ID}/files"

FILES = {
    "consonant_p1": "NSL_Consonant_Part_1.zip",
    "consonant_p2": "NSL_Consonant_Part_2.zip",
    "consonant_p3": "NSL_Consonant_Part_3.zip",
    "vowel": "NSL_Vowel.zip",
}

# Choose what to download
DOWNLOAD_WHICH = ["consonant_p1", "consonant_p2", "consonant_p3"]  # edit if needed

def download_file(fname):
    url = f"{BASE}/{fname}?download=1"
    if Path(fname).exists():
        print(f"Already exists: {fname}")
        return
    print("Downloading:", fname)
    # -c continue, -q quiet-ish
    !wget -c -O "{fname}" "{url}"

def unzip_file(zipname):
    outdir = zipname.replace(".zip","")
    if Path(outdir).exists():
        print(f"Already unzipped: {outdir}/")
        return
    print("Unzipping:", zipname)
    !unzip -q "{zipname}" -d "{outdir}"

for key in DOWNLOAD_WHICH:
    download_file(FILES[key])
    unzip_file(FILES[key])

print("Done.")


Downloading: NSL_Consonant_Part_1.zip
--2026-01-06 01:54:21--  https://zenodo.org/records/10478554/files/NSL_Consonant_Part_1.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.48.75, 137.138.52.235, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|188.185.48.75|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3455940835 (3.2G) [application/octet-stream]
Saving to: ‘NSL_Consonant_Part_1.zip’

NSL_Consonant_Part_ 100%[===================>]   3.22G  7.89MB/s    in 29m 8s  

2026-01-06 02:23:29 (1.89 MB/s) - ‘NSL_Consonant_Part_1.zip’ saved [3455940835/3455940835]

Unzipping: NSL_Consonant_Part_1.zip
Downloading: NSL_Consonant_Part_2.zip
--2026-01-06 02:24:00--  https://zenodo.org/records/10478554/files/NSL_Consonant_Part_2.zip?download=1
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 188.185.48.75, 137.138.52.235, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Leng

## 3) Build an index of video clips + labels

NSL23 videos are stored in nested folders and filenames look like:
- `S1_BA.MOV`, `S12_ANGA.MOV`, etc.
We parse:
- **Signer**: prefix `S<number>` (e.g., `S1`)
- **Label**: the rest of the stem (e.g., `BA`, `D_SA`, `ANGA`)

We also **skip aggregated recordings** where labels include tokens like `NSL`, `ALL`, `PHONE`.


In [ ]:
# =========================
# 3) Indexing utilities (NSL23-robust)
# =========================
# NSL23 details (from dataset documentation):
# - 14 volunteers (S1..S14), right hand
# - 36 consonants + 13 vowels (some vowels are dynamic)
# - Filenames sometimes include repeats like "A (2).mov" and not always prefixed by signer.
# - Dataset also contains long "ALL" recordings; we exclude those by name + duration threshold.

SIGNER_RE = re.compile(r"\b(S\d{1,2})\b", re.IGNORECASE)

def _extract_signer_from_path(path: Path):
    # Prefer signer from folder name (e.g., .../S1/A (2).mov) or anywhere in path
    for part in [path.parent.name] + list(path.parts[::-1]):
        m = SIGNER_RE.search(str(part))
        if m:
            return m.group(1).upper()
    return "UNK"

def clean_label(raw: str):
    """
    Normalize label from filename stem:
    - remove repeat suffixes like "(2)" or "(3)"
    - remove trailing digits/underscores
    - collapse spaces to underscores
    """
    s = raw.upper().strip()
    # remove signer prefix if present like S1_
    s = re.sub(r"^S\d{1,2}_", "", s)
    # remove parenthetical segments like "(2)" or "(3)"
    s = re.sub(r"\([^)]*\)", "", s)
    # remove trailing digits and separators
    s = re.sub(r"[\s_\-]*\d+$", "", s)
    s = s.replace(" ", "_")
    s = re.sub(r"__+", "_", s).strip("_")
    # common alias fixes seen in NSL23 listings
    if s in {"KSH", "KSHHA", "K_SH", "K_SHA"}:
        s = "KSHA"
    return s

def parse_label_and_signer(path: Path):
    stem = path.stem
    signer = _extract_signer_from_path(path)
    label = clean_label(stem)
    return label, signer

def looks_like_aggregate(label: str):
    # Aggregate/long recordings typically contain these tokens
    bad = ["NSL", "ALL", "PHONE", "CAMERA", "CONSONANT", "VOWEL", "REALWORLD", "REAL_WORLD", "UNPREPARED", "PREPARED"]
    return any(tok in label for tok in bad)

def video_duration_seconds(path: Path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        return None
    fps = cap.get(cv2.CAP_PROP_FPS) or 0.0
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0.0
    cap.release()
    if fps <= 1e-6 or frames <= 0:
        return None
    return float(frames / fps)

def collect_videos(root: Path, prefer_cropped=True, max_duration_s=10.0):
    exts = {".mov", ".MOV", ".mp4", ".MP4"}
    all_paths = [p for p in root.rglob("*") if p.suffix in exts]

    if prefer_cropped:
        cropped = [p for p in all_paths if "cropped" in str(p).lower()]
        if len(cropped) > 0:
            all_paths = cropped
            print(f"Using cropped videos only: {len(all_paths)} files")
        else:
            print(f"No cropped folders found; using all videos: {len(all_paths)} files")

    items = []
    dropped_agg = 0
    dropped_long = 0

    for p in all_paths:
        label, signer = parse_label_and_signer(p)
        if looks_like_aggregate(label):
            dropped_agg += 1
            continue

        # Drop very long recordings (aggregate sequences). Individual signs are ~1–5 seconds in dataset tables.
        dur = video_duration_seconds(p)
        if dur is not None and dur > max_duration_s:
            dropped_long += 1
            continue

        items.append({"path": str(p), "label": label, "signer": signer, "duration_s": dur})

    print(f"Dropped by name (aggregate): {dropped_agg}")
    print(f"Dropped by duration > {max_duration_s}s: {dropped_long}")
    return items


In [ ]:
# Optional: visualize a few examples from the index
def show_random_samples(items, k=6):
    chosen = random.sample(items, k=min(k, len(items)))
    fig, axes = plt.subplots(1, len(chosen), figsize=(3*len(chosen), 3))
    if len(chosen)==1:
        axes=[axes]
    for ax, it in zip(axes, chosen):
        p = Path(it["path"])
        cap = cv2.VideoCapture(str(p))
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        ret, frame = cap.read()
        cap.release()
        if not ret:
            ax.set_title(f"{it['label']} (fail)")
            ax.axis("off")
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        ax.imshow(frame)
        ax.set_title(f"{it['label']} / {it['signer']}")
        ax.axis("off")
    plt.show()

show_random_samples(items, k=6)


NameError: name 'items' is not defined

## Save Data to Google Drive (Optional)

To persist the downloaded NSL23 dataset across sessions, you can move it to your Google Drive. This prevents you from re-downloading large files every time you open the notebook.

## 4) Train/Val/Test Split (Signer-aware)

To avoid leaking signer identity, we do a **grouped split by signer** using `GroupShuffleSplit`.


In [ ]:
# =========================
# 4) Split by signer (grouped)
# =========================
def grouped_split(items, test_size=0.1, val_size=0.1, seed=42):
    labels = np.array([it["label"] for it in items])
    groups = np.array([it["signer"] for it in items])
    idx = np.arange(len(items))

    gss1 = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trainval_idx, test_idx = next(gss1.split(idx, labels, groups))

    # Now split trainval into train and val
    trainval_items = [items[i] for i in trainval_idx]
    trainval_labels = np.array([it["label"] for it in trainval_items])
    trainval_groups = np.array([it["signer"] for it in trainval_items])
    trainval_idx2 = np.arange(len(trainval_items))

    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size/(1-test_size), random_state=seed)
    train_idx2, val_idx2 = next(gss2.split(trainval_idx2, trainval_labels, trainval_groups))

    train = [trainval_items[i] for i in train_idx2]
    val = [trainval_items[i] for i in val_idx2]
    test = [items[i] for i in test_idx]
    return train, val, test

train_items, val_items, test_items = grouped_split(items, test_size=0.1, val_size=0.1, seed=42)
print("Train/Val/Test:", len(train_items), len(val_items), len(test_items))

# Build label mapping from TRAIN only (safe)
train_labels = sorted(set([it["label"] for it in train_items]))
label2idx = {lab:i for i,lab in enumerate(train_labels)}
idx2label = {i:lab for lab,i in label2idx.items()}
print("Num train classes:", len(label2idx))

# Filter out val/test clips with unseen labels (should be rare, but safe)
def filter_seen(items, label2idx):
    out = [it for it in items if it["label"] in label2idx]
    dropped = len(items)-len(out)
    if dropped:
        print("Dropped unseen-label clips:", dropped)
    return out

val_items = filter_seen(val_items, label2idx)
test_items = filter_seen(test_items, label2idx)


## 5) Video loading + Dataset

We sample a fixed number of frames `NUM_FRAMES` uniformly from each video using OpenCV (fast + works on `.mov`).  
Each example returns a tensor of shape: **(T, 3, H, W)** where T=`NUM_FRAMES`.

We use ImageNet normalization for the CNN backbone.


In [ ]:
# =========================
# 5) Dataset
# =========================
IMG_SIZE = 224
NUM_FRAMES = 16  # adjust (16 or 24 are common)
BATCH_SIZE = 8

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

def sample_frames_cv2(path: str, num_frames=16, size=224):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {path}")
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        frames=[]
        while True:
            ret, frame = cap.read()
            if not ret: break
            frames.append(frame)
        total=len(frames)
        idxs=np.linspace(0, total-1, num_frames).astype(int) if total>0 else np.zeros(num_frames, dtype=int)
        sel=[frames[i] if total>0 else np.zeros((size,size,3),dtype=np.uint8) for i in idxs]
    else:
        idxs = np.linspace(0, total-1, num_frames).astype(int)
        sel=[]
        for i in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
            ret, frame = cap.read()
            if not ret:
                frame = sel[-1] if sel else np.zeros((size,size,3), dtype=np.uint8)
            sel.append(frame)
    cap.release()
    out=[]
    for frame in sel:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (size,size))
        out.append(frame)
    arr = np.stack(out).astype(np.float32)/255.0  # (T,H,W,3)
    x = torch.from_numpy(arr).permute(0,3,1,2)    # (T,3,H,W)
    return x

class NSL23VideoDataset(Dataset):
    def __init__(self, items, label2idx, num_frames=16, img_size=224, training=True):
        self.items = items
        self.label2idx = label2idx
        self.num_frames = num_frames
        self.img_size = img_size
        self.training = training

    def __len__(self):
        return len(self.items)

    def _augment(self, x):
        # x: (T,3,H,W)
        # Avoid horizontal flip by default (can change meaning). Use mild brightness/contrast jitter.
        if random.random() < 0.3:
            # brightness
            delta = (random.random()*0.2 - 0.1)
            x = torch.clamp(x + delta, 0.0, 1.0)
        if random.random() < 0.3:
            # contrast
            factor = 1.0 + (random.random()*0.3 - 0.15)
            mean = x.mean(dim=(2,3), keepdim=True)
            x = torch.clamp((x - mean) * factor + mean, 0.0, 1.0)
        return x

    def __getitem__(self, idx):
        it = self.items[idx]
        x = sample_frames_cv2(it["path"], num_frames=self.num_frames, size=self.img_size)  # (T,3,H,W) in [0,1]
        if self.training:
            x = self._augment(x)
        # normalize per frame
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        y = self.label2idx[it["label"]]
        return x, y

train_ds = NSL23VideoDataset(train_items, label2idx, num_frames=NUM_FRAMES, img_size=IMG_SIZE, training=True)
val_ds   = NSL23VideoDataset(val_items,   label2idx, num_frames=NUM_FRAMES, img_size=IMG_SIZE, training=False)
test_ds  = NSL23VideoDataset(test_items,  label2idx, num_frames=NUM_FRAMES, img_size=IMG_SIZE, training=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Smoke test a batch
xb, yb = next(iter(train_loader))
print("Batch X:", xb.shape, xb.dtype, "Batch y:", yb.shape)


## 6) Model: CNN + Transformer Encoder

- CNN backbone: **MobileNetV2** (ImageNet-pretrained)
- Per-frame feature vector projected to `D_MODEL`
- Transformer encoder over the **time dimension**
- Mean-pooling over time, then linear classifier


In [ ]:
# =========================
# 6) Model
# =========================
# Use torchvision if available (Colab has it). If torchvision import fails in some environments,
# you can fallback to timm; but in Colab this should work.
import torchvision

class CNNTransformer(nn.Module):
    def __init__(self, num_classes, d_model=512, nhead=8, num_layers=4, ff_dim=2048, dropout=0.1, max_len=256):
        super().__init__()
        backbone = torchvision.models.mobilenet_v2(weights=torchvision.models.MobileNet_V2_Weights.DEFAULT)
        self.cnn = backbone.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.feat_dim = backbone.last_channel  # 1280

        self.proj = nn.Linear(self.feat_dim, d_model)
        self.pos = nn.Parameter(torch.zeros(max_len, d_model))
        nn.init.trunc_normal_(self.pos, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        # x: (B,T,3,H,W)
        B,T,C,H,W = x.shape
        x = x.view(B*T, C, H, W)
        f = self.cnn(x)                # (B*T, ch, h', w')
        f = self.pool(f).flatten(1)    # (B*T, feat_dim)
        f = self.proj(f)               # (B*T, d_model)
        f = f.view(B, T, -1)           # (B,T,d_model)

        # add positional embeddings
        f = f + self.pos[:T].unsqueeze(0)
        z = self.transformer(f)        # (B,T,d_model)
        z = self.norm(z)
        z = z.mean(dim=1)              # (B,d_model)
        return self.head(z)

NUM_CLASSES = len(label2idx)
model = CNNTransformer(num_classes=NUM_CLASSES, d_model=512, nhead=8, num_layers=4).to(DEVICE)
print("Model params:", sum(p.numel() for p in model.parameters())/1e6, "M")


## 7) Training

We train with:
- Cross-entropy loss
- AdamW optimizer
- AMP mixed precision (if GPU)
- Best checkpoint saved by validation accuracy


In [ ]:
# =========================
# 7) Training loop
# =========================
EPOCHS = 15
LR = 1e-4
WD = 1e-4

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

def run_one_epoch(model, loader, train=True):
    model.train(train)
    total_loss = 0.0
    preds_all, y_all = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=True)
        yb = yb.to(DEVICE, non_blocking=True)
        if train:
            optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE.type=="cuda")):
            logits = model(xb)
            loss = criterion(logits, yb)
        if train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item() * xb.size(0)
        preds = torch.argmax(logits, dim=1)
        preds_all.append(preds.detach().cpu())
        y_all.append(yb.detach().cpu())

    total_loss /= len(loader.dataset)
    preds_all = torch.cat(preds_all).numpy()
    y_all = torch.cat(y_all).numpy()
    acc = accuracy_score(y_all, preds_all)
    f1 = f1_score(y_all, preds_all, average="macro")
    return total_loss, acc, f1

best_val_acc = 0.0
ckpt_path = Path("/content/best_cnn_transformer.pt")

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc, tr_f1 = run_one_epoch(model, train_loader, train=True)
    va_loss, va_acc, va_f1 = run_one_epoch(model, val_loader, train=False)
    scheduler.step()

    print(f"Epoch {epoch:02d} | "
          f"train loss {tr_loss:.4f} acc {tr_acc:.4f} f1 {tr_f1:.4f} | "
          f"val loss {va_loss:.4f} acc {va_acc:.4f} f1 {va_f1:.4f}")

    if va_acc > best_val_acc:
        best_val_acc = va_acc
        torch.save({
            "model": model.state_dict(),
            "label2idx": label2idx,
            "idx2label": idx2label,
            "epoch": epoch,
            "val_acc": va_acc,
        }, ckpt_path)
        print("  ✅ saved best checkpoint:", ckpt_path)

print("Best val acc:", best_val_acc)


## 8) Evaluation on Test Set


In [ ]:
# =========================
# 8) Test evaluation
# =========================
ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.append(preds)
        all_true.append(yb.numpy())

y_true = np.concatenate(all_true)
y_pred = np.concatenate(all_preds)

print("Test Accuracy:", accuracy_score(y_true, y_pred))
print("Test Macro F1:", f1_score(y_true, y_pred, average="macro"))

print("\nClassification report:\n")
print(classification_report(y_true, y_pred, target_names=[idx2label[i] for i in range(NUM_CLASSES)]))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10,10))
plt.imshow(cm, interpolation='nearest')
plt.title("Confusion Matrix")
plt.colorbar()
plt.tight_layout()
plt.show()


## 9) Inference on a Single Video

Point `VIDEO_PATH` to any `.mov` in the dataset and run.


In [ ]:
def predict_one_video(video_path: str):
    model.eval()
    x = sample_frames_cv2(video_path, num_frames=NUM_FRAMES, size=IMG_SIZE)
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    x = x.unsqueeze(0).to(DEVICE)  # (1,T,3,H,W)
    with torch.no_grad():
        logits = model(x)
        prob = torch.softmax(logits, dim=1)[0]
        top = torch.topk(prob, k=min(5, prob.numel()))
    return [(idx2label[i.item()], top.values[j].item()) for j,i in enumerate(top.indices)]

# Pick a random test clip
sample = random.choice(test_items)
print("Video:", sample["path"])
print("True label:", sample["label"])
print("Top predictions:", predict_one_video(sample["path"]))


## 10) (Optional) Save a True Dataset Sample Grid for Your Paper

This cell saves a grid image from real NSL23 videos (first frame of several clips).
You can use it in your LaTeX paper instead of placeholders.


In [ ]:
def save_sample_grid(items, out_path="nsl23_samples_grid.png", n=12):
    chosen = random.sample(items, k=min(n, len(items)))
    cols = 6
    rows = math.ceil(len(chosen)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*3, rows*3))
    axes = np.array(axes).reshape(rows, cols)
    for ax in axes.flatten():
        ax.axis("off")
    for i, it in enumerate(chosen):
        r, c = divmod(i, cols)
        p = it["path"]
        cap = cv2.VideoCapture(p)
        ret, frame = cap.read()
        cap.release()
        if not ret:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        axes[r,c].imshow(frame)
        axes[r,c].set_title(it["label"], fontsize=10)
        axes[r,c].axis("off")
    plt.tight_layout()
    plt.savefig(out_path, dpi=200)
    print("Saved:", out_path)

save_sample_grid(items, out_path="/content/nsl23_samples_grid.png", n=12)


## 11) Export to GitHub (Suggested)

If you plan to link a GitHub repo in the paper instead of embedding code:
- include `requirements.txt` (or `environment.yml`)
- include this notebook and a short `README.md`
- add a `models/` folder for checkpoints
- add a `data/` note explaining that NSL23 is downloaded from Zenodo (and not redistributed)

This is commonly preferred by reviewers, as it keeps the manuscript cleaner while ensuring reproducibility.


# 12) Benchmark Suite (Baselines + Results Table)

To justify claims like “outperforming previous CNN-based approaches,” you should report **apples-to-apples baselines on the same NSL23 split**.

This section trains and compares:
- **Baseline A: Single-frame CNN** (no temporal modeling)
- **Baseline B: CNN + BiLSTM** (classic temporal model)
- **Proposed: CNN + Transformer** (your main model)

Outputs:
- Test Accuracy + Macro-F1
- Simple speed estimate (FPS) on your hardware
- A results table (copy/paste into paper)

> Tip: Set `BENCH_EPOCHS` small (e.g., 5) for a quick check, then increase for final numbers.


In [ ]:
# =========================
# 12.1) Baseline models
# =========================
import time
import pandas as pd

class SingleFrameCNN(nn.Module):
    """Baseline A: single-frame classifier.
    Uses the same input (B,T,3,H,W) but only uses the middle frame."""
    def __init__(self, num_classes):
        super().__init__()
        backbone = torchvision.models.mobilenet_v2(weights=torchvision.models.MobileNet_V2_Weights.DEFAULT)
        self.features = backbone.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        feat_dim = backbone.last_channel  # 1280
        self.classifier = nn.Linear(feat_dim, num_classes)

    def forward(self, x):
        B,T,_,_,_ = x.shape
        frame = x[:, T//2]  # (B,3,H,W)
        f = self.features(frame)
        f = self.pool(f).flatten(1)
        return self.classifier(f)

class CNNBiLSTM(nn.Module):
    """Baseline B: CNN features per frame + BiLSTM over time."""
    def __init__(self, num_classes, d_model=512, hidden=256, num_layers=1, dropout=0.1):
        super().__init__()
        backbone = torchvision.models.mobilenet_v2(weights=torchvision.models.MobileNet_V2_Weights.DEFAULT)
        self.cnn = backbone.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.feat_dim = backbone.last_channel  # 1280

        self.proj = nn.Linear(self.feat_dim, d_model)
        self.lstm = nn.LSTM(
            input_size=d_model,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.norm = nn.LayerNorm(hidden*2)
        self.head = nn.Linear(hidden*2, num_classes)

    def forward(self, x):
        B,T,C,H,W = x.shape
        x = x.view(B*T, C, H, W)
        f = self.cnn(x)
        f = self.pool(f).flatten(1)         # (B*T, feat_dim)
        f = self.proj(f).view(B, T, -1)     # (B,T,d_model)
        out, _ = self.lstm(f)               # (B,T,2*hidden)
        out = self.norm(out)
        z = out.mean(dim=1)                 # mean pool over time
        return self.head(z)


In [ ]:
# =========================
# 12.2) Training / eval helpers
# =========================
from sklearn.metrics import f1_score, accuracy_score

def measure_fps(model, loader, warmup=5, iters=20):
    """Rough throughput estimate using one batch."""
    model.eval()
    xb, _ = next(iter(loader))
    xb = xb.to(DEVICE)
    B,T,_,_,_ = xb.shape

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(xb)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(iters):
            _ = model(xb)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
    end = time.perf_counter()

    total_videos = B * iters
    total_frames = B * T * iters
    seconds = end - start
    fps_video = total_videos / seconds
    ms_per_frame = (seconds / total_frames) * 1000.0
    return fps_video, ms_per_frame

def train_model(model, name, train_loader, val_loader, epochs=10, lr=1e-4, wd=1e-4):
    model = model.to(DEVICE)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type=="cuda"))

    best = {"val_acc": 0.0, "state": None, "epoch": 0}
    for epoch in range(1, epochs+1):
        model.train()
        tr_loss, tr_preds, tr_true = 0.0, [], []
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast(enabled=(DEVICE.type=="cuda")):
                logits = model(xb)
                loss = crit(logits, yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            tr_loss += loss.item() * xb.size(0)
            tr_preds.append(torch.argmax(logits, dim=1).detach().cpu())
            tr_true.append(yb.detach().cpu())
        sch.step()

        tr_loss /= len(train_loader.dataset)
        tr_preds = torch.cat(tr_preds).numpy()
        tr_true  = torch.cat(tr_true).numpy()
        tr_acc = accuracy_score(tr_true, tr_preds)
        tr_f1  = f1_score(tr_true, tr_preds, average="macro")

        model.eval()
        va_loss, va_preds, va_true = 0.0, [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                logits = model(xb)
                loss = crit(logits, yb)
                va_loss += loss.item() * xb.size(0)
                va_preds.append(torch.argmax(logits, dim=1).cpu())
                va_true.append(yb.cpu())
        va_loss /= len(val_loader.dataset)
        va_preds = torch.cat(va_preds).numpy()
        va_true  = torch.cat(va_true).numpy()
        va_acc = accuracy_score(va_true, va_preds)
        va_f1  = f1_score(va_true, va_preds, average="macro")

        print(f"[{name}] Epoch {epoch:02d}/{epochs} | "
              f"train loss {tr_loss:.4f} acc {tr_acc:.4f} f1 {tr_f1:.4f} | "
              f"val loss {va_loss:.4f} acc {va_acc:.4f} f1 {va_f1:.4f}")

        if va_acc > best["val_acc"]:
            best = {"val_acc": va_acc, "state": {k:v.detach().cpu() for k,v in model.state_dict().items()}, "epoch": epoch}

    model.load_state_dict(best["state"])
    return model, best

def eval_model(model, loader):
    model.eval()
    preds, true = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            preds.append(torch.argmax(logits, dim=1).cpu().numpy())
            true.append(yb.numpy())
    y_true = np.concatenate(true)
    y_pred = np.concatenate(preds)
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "y_true": y_true,
        "y_pred": y_pred
    }


In [ ]:
# =========================
# 12.3) Run benchmark suite
# =========================
BENCH_EPOCHS = 8        # increase for final paper numbers (e.g., 15-30)
BENCH_LR = 1e-4
BENCH_WD = 1e-4

RUN_MODELS = [
    "single_frame_cnn",
    "cnn_bilstm",
    "cnn_transformer",
]

trained_models = {}
results = []

def build_model(key):
    if key == "single_frame_cnn":
        return SingleFrameCNN(num_classes=NUM_CLASSES), "Single-frame CNN (MobileNetV2)"
    if key == "cnn_bilstm":
        return CNNBiLSTM(num_classes=NUM_CLASSES, d_model=512, hidden=256), "CNN + BiLSTM"
    if key == "cnn_transformer":
        return CNNTransformer(num_classes=NUM_CLASSES, d_model=512, nhead=8, num_layers=4), "CNN + Transformer (Proposed)"
    raise ValueError(key)

for key in RUN_MODELS:
    model_i, name = build_model(key)
    model_i, best = train_model(model_i, name, train_loader, val_loader,
                                epochs=BENCH_EPOCHS, lr=BENCH_LR, wd=BENCH_WD)
    metrics = eval_model(model_i, test_loader)
    fps_video, ms_per_frame = measure_fps(model_i, test_loader)

    trained_models[name] = model_i  # keep model for confusion matrix etc.

    results.append({
        "Model": name,
        "Best Val Acc": best["val_acc"],
        "Best Epoch": best["epoch"],
        "Test Acc": metrics["acc"],
        "Test Macro-F1": metrics["macro_f1"],
        "FPS (videos/s, batch)": fps_video,
        "ms/frame (batch)": ms_per_frame,
    })

df = pd.DataFrame(results).sort_values("Test Acc", ascending=False)
df


In [ ]:
# =========================
# 12.4) Confusion matrix for the best model + save table
# =========================
from sklearn.metrics import confusion_matrix

out_csv = "/content/nsl23_benchmark_results.csv"
df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

best_name = df.iloc[0]["Model"]
best_model = trained_models[best_name]
print("Best model:", best_name)

metrics = eval_model(best_model, test_loader)
cm = confusion_matrix(metrics["y_true"], metrics["y_pred"])

plt.figure(figsize=(10,10))
plt.imshow(cm, interpolation='nearest')
plt.title(f"Confusion Matrix — {best_name}")
plt.colorbar()
plt.tight_layout()
plt.show()


## 12.5) Reporting guardrail

This legacy notebook is not the authoritative paper pipeline. Do not use it to make a state-of-the-art, deployment, continuous-recognition, or real-time-system claim. The verified paper result comes from the frozen two-fold scripts and artifacts in the repository. A defensible description is:

> “In an exploratory isolated-clip evaluation with two held-out development signers, the best preregistered RGB baseline was CNN–BiLSTM. Runtime measurements are device- and pipeline-specific engineering observations, not evidence of live or continuous NSL recognition.”

Use the manuscript and CLAIM_LEDGER.md for exact numbers and limitations.
